# 03 · Train ResNet-18 (A or B)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/03_train_resnet.ipynb)

**Runtime → T4 GPU.** Выбери `CONFIG` и `SEEDS` в ячейке ниже и запусти всё.

Порядок работы:
1. `resnet_A`, `SEEDS = [0]` — проверить, что обучение сходится; при необходимости поменять `epochs` / `lr`
   в **обоих** `configs/resnet_A.yaml` и `configs/resnet_B.yaml` (они должны отличаться только `p_swap`) и запушить.
2. Когда конфиг зафиксирован: `resnet_A` с `SEEDS = [0, 1, 2]`, затем `resnet_B` с `SEEDS = [0, 1, 2]`.

Checkpoints: `MyDrive/bgbias/runs/<name>_seed<k>/` (`best.pt` по val original, `last.pt` для продолжения).
Если Colab отключился — просто запусти ноутбук снова, обучение продолжится с последней эпохи.

In [ ]:
CONFIG = "resnet_A"   # "resnet_A" or "resnet_B"
SEEDS = [0]           # after the config is fixed: [0, 1, 2]

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
REPO_URL = "https://github.com/mundmina/dl_project.git"
REPO_DIR = "/content/dl_project"
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:  # always run the latest code from GitHub
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
from bgbias.train import load_config, train

cfg = load_config(f"{REPO_DIR}/configs/{CONFIG}.yaml")
cfg

In [ ]:
import pandas as pd

histories = {}
for seed in SEEDS:
    run_dir = RUNS / f"{cfg['name']}_seed{seed}"
    print(f"=== {run_dir.name} ===")
    histories[seed] = pd.DataFrame(train(cfg, seed, data, run_dir))

In [ ]:
for seed, h in histories.items():
    best = h.loc[h.val_acc_original.idxmax()]
    print(f"seed {seed}: best epoch {int(best.epoch)}, val original {best.val_acc_original:.4f}, "
          f"val swapped {best.val_acc_swapped:.4f}, gap {best.val_acc_original - best.val_acc_swapped:.4f}")
pd.concat(histories, names=["seed"]).round(4)